# DOE MASTER ML PIPELINE — target contracts, paper-aligned models, clean outputs

This notebook rebuilds the North Slope gas-hydrate ML workflow around the actual research questions:

1. **Hydrate occurrence** — classification (`none`, `pore-filling`, `fracture-filling`) only when independently interpreted labels are present.
2. **Hydrate saturation** — regression against one explicitly documented reference saturation per well.

It does **not** treat water saturation as a project target, and it does **not** merge `Sgh`, `S_h`, `Sh`, or `NMR_SAT` merely because their names look similar. Every target-like source column is retained separately until its definition, units, derivation, and role are approved.

## Paper alignment

The workflow follows the structure of the source ML studies while correcting the main validation and provenance risks:

- routine well-log feature families used in Singh et al. (2021), Chong et al. (2022), and Chong et al. (2024);
- min–max feature scaling fitted on training data only;
- ANN-style models plus simpler benchmark algorithms;
- an independently held-out **blind well** for routine development;
- balanced metrics for occurrence classification;
- explicit sensitivity flags when a predictor participates in the target formula.

The source papers repeatedly sampled depth rows and often repeated training 100 times. This notebook defaults to **one fixed split and one realization** so ordinary development stays understandable. Repeats and full well rotation are optional final audits, not routine output generators.

## Runtime outputs

Normal operation writes at most four files:

```text
outputs_runtime/ml_master/model_results.xlsx
outputs_runtime/ml_master/predictions.parquet
outputs_runtime/ml_master/paper_figures.pdf
models_runtime/ml_master/selected_models.joblib
```

In `RUN_MODE = "audit"`, only `model_results.xlsx` is written.

## 0. Configuration — edit only this section

Run the notebook first in `audit` mode. Review the `target_catalog`, `target_pair_checks`, and `target_contract` sheets. Then enter the exact approved source header for each well, mark the contract approved, and switch to `model` mode.

A source header such as `Sh` is notation, not provenance. It must not be approved until the workbook description, formula, or accompanying source documentation establishes what produced it.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
from typing import Any, Iterable
import json
import math
import os
import re
import warnings

import joblib
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import (
    AdaBoostRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import LogisticRegression, Ridge, SGDRegressor
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_recall_fscore_support,
    r2_score,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures, SplineTransformer
from sklearn.svm import SVC, SVR
from sklearn.tree import DecisionTreeRegressor

# -----------------------------------------------------------------------------
# Source files
# -----------------------------------------------------------------------------
DEFAULT_DATA_DIR = Path.home() / "Downloads" / "Northslopedatasets06052026"
DATA_DIR = Path(os.environ.get("NORTH_SLOPE_DATA_DIR", DEFAULT_DATA_DIR)).expanduser()

SOURCE_SPECS: list[dict[str, Any]] = [
    {
        "well_alias": "WellA",
        "filename": "2L-38_input_Mallik.xlsx",
        "sheet_name": None,
        "source_label": "Mallik 2L-38",
        "default_depth_unit": "m",
        "default_density_unit": "kg/m3",
        "default_caliper_unit": "mm",
    },
    {
        "well_alias": "WellB",
        "filename": "5L-38_input_Mallik.xlsx",
        "sheet_name": None,
        "source_label": "Mallik 5L-38",
        "default_depth_unit": "m",
        "default_density_unit": "kg/m3",
        "default_caliper_unit": "mm",
    },
    {
        "well_alias": "WellC",
        "filename": "MtElbert_Ignik_input_ANS.xlsx",
        "sheet_name": "MTE",
        "source_label": "Mount Elbert",
        "default_depth_unit": "ft",
        "default_density_unit": "g/cc",
        "default_caliper_unit": "in",
    },
    {
        "well_alias": "WellD",
        "filename": "MtElbert_Ignik_input_ANS.xlsx",
        "sheet_name": "IGS",
        "source_label": "Iġnik Sikumi",
        "default_depth_unit": "ft",
        "default_density_unit": "g/cc",
        "default_caliper_unit": "in",
    },
]

# -----------------------------------------------------------------------------
# Run controls
# -----------------------------------------------------------------------------
RUN_MODE = "audit"  # "audit" first; change to "model" only after target approval
RANDOM_STATE = 42
N_REPEATS = 1       # source-paper replication can use 100 after the workflow is frozen
RUN_FINAL_WELL_ROTATION = False

TRAIN_WELLS = ("WellA", "WellB", "WellC")
BLIND_WELL = "WellD"

# The compact ladder keeps routine runs understandable. Set to "paper_12" only
# for the final algorithm benchmark.
MODEL_SET = "compact"  # "compact" or "paper_12"
PRIMARY_FEATURE_PANEL = "chong_permafrost_3"
RUN_FEATURE_PANEL_SENSITIVITY = False
BALANCE_TRAINING_WELLS = True
BALANCE_OCCURRENCE_CLASSES = True
MIN_TARGET_ROWS_PER_WELL = 20
MIN_FEATURE_COVERAGE_TRAIN = 0.20
MIN_ROW_FEATURE_FRACTION = 0.75
CLIP_SATURATION_TO_UNIT_INTERVAL = True
REFIT_SELECTED_MODEL_ON_ALL_LABELED_WELLS = True

# -----------------------------------------------------------------------------
# Explicit target contracts
# -----------------------------------------------------------------------------
# Enter exact source header strings after reviewing the audit output. Do not use
# lists of aliases here. Each well must point to one documented target column.
TARGET_CONTRACTS: dict[str, dict[str, Any]] = {
    "hydrate_saturation": {
        "enabled": True,
        "task": "regression",
        "approved": False,
        "expected_family": "hydrate_saturation",
        "required_provenance": "Documented NMR-density, Archie, acoustic, or other named reference method",
        # Required before approval. Examples: nmr_density, archie_resistivity,
        # acoustic_velocity, core_calibrated, other_documented.
        "reference_method": None,
        "target_definition": None,  # exact equation or interpretation rule
        "source_document": None,    # workbook sheet/formula cell or report citation
        "source_header_by_well": {
            "WellA": None,
            "WellB": None,
            "WellC": None,
            "WellD": None,
        },
        # List predictors directly used in the approved target formula. These
        # are removed under strict feature policy and retained only in a clearly
        # marked paper-replication sensitivity run.
        "formula_input_features": [],
    },
    "hydrate_occurrence": {
        "enabled": False,
        "task": "classification",
        "approved": False,
        "expected_family": "hydrate_occurrence",
        "required_provenance": "Independent RAB/core/multi-log interpretation; never thresholded from saturation",
        "reference_method": None,   # e.g., rab_interpretation or core_log_interpretation
        "target_definition": None,  # class definitions and labeling procedure
        "source_document": None,
        "independent_from_saturation": False,
        # Record logs used by the human interpretation. They are not silently
        # removed, but their overlap with model predictors is disclosed.
        "labeling_evidence_features": [],
        "source_header_by_well": {
            "WellA": None,
            "WellB": None,
            "WellC": None,
            "WellD": None,
        },
        "formula_input_features": [],
    },
}

# "strict" removes direct formula inputs from the predictor matrix.
# "paper_replication" permits them but records the circularity sensitivity.
FEATURE_POLICY = "strict"  # "strict" or "paper_replication"

FEATURE_PANELS: dict[str, tuple[str, ...]] = {
    # Singh et al. proposed porosity + bulk density + compressional velocity.
    "singh_3": ("neutron_porosity_vv", "rhob_g_cc", "vp_m_s"),
    # Chong et al. reported strong permafrost transfer with porosity + Rt + Vp.
    "chong_permafrost_3": ("density_porosity_vv", "rt_ohm_m", "vp_m_s"),
    "chong_vp_gr": ("vp_m_s", "gr_api"),
    # Occurrence paper feature family: rho, Rt, GR, Vp, density porosity.
    "occurrence_5": ("rhob_g_cc", "rt_ohm_m", "gr_api", "vp_m_s", "density_porosity_vv"),
    "routine_6": (
        "gr_api", "rhob_g_cc", "neutron_porosity_vv",
        "density_porosity_vv", "rt_ohm_m", "vp_m_s",
    ),
    "routine_7_with_vs": (
        "gr_api", "rhob_g_cc", "neutron_porosity_vv",
        "density_porosity_vv", "rt_ohm_m", "vp_m_s", "vs_m_s",
    ),
}

# -----------------------------------------------------------------------------
# Clean outputs: stable names, no per-fold directories
# -----------------------------------------------------------------------------
WORKSPACE_ROOT = Path(os.environ.get("NORTH_SLOPE_WORKSPACE", Path.cwd())).expanduser()
OUTPUT_DIR = WORKSPACE_ROOT / "outputs_runtime" / "ml_master"
MODEL_DIR = WORKSPACE_ROOT / "models_runtime" / "ml_master"
RESULTS_XLSX = OUTPUT_DIR / "model_results.xlsx"
PREDICTIONS_PARQUET = OUTPUT_DIR / "predictions.parquet"
FIGURES_PDF = OUTPUT_DIR / "paper_figures.pdf"
SELECTED_MODELS = MODEL_DIR / "selected_models.joblib"

PAPER_REFERENCES = {
    "Singh_2021": "doi:10.1007/s10596-020-10004-3",
    "Chong_2022": "doi:10.1007/s10596-022-10151-9",
    "Chong_2024": "doi:10.1190/INT-2023-0046.1",
}

PAPER_TARGET_METHODS = [
    {
        "paper": "Singh et al. (2021)",
        "task": "regression",
        "paper_target": "gas-hydrate saturation (Sh)",
        "reference_method": "NMR-density porosity",
        "ground_truth_basis": "Sh = 1 - phi_NMR / phi_density",
        "target_formula_inputs": "NMR porosity; density porosity",
        "model_inputs_reported": "neutron porosity; bulk density; Vp (primary three-log proposal)",
        "audit_implication": "Do not equate a column named Sh with NMR-density Sh unless workbook provenance confirms it.",
    },
    {
        "paper": "Chong et al. (2022)",
        "task": "regression",
        "paper_target": "gas-hydrate saturation (Sgh)",
        "reference_method": "NMR-density porosity",
        "ground_truth_basis": "NMR-derived Sgh; resistivity used as qualitative presence confirmation",
        "target_formula_inputs": "NMR porosity; density porosity",
        "model_inputs_reported": "rho; density porosity; GR; Rt; Vp; Vs in tested combinations",
        "audit_implication": "A paper-replication run may overlap target-formula inputs; strict runs must disclose or remove them.",
    },
    {
        "paper": "Chong et al. (2024)",
        "task": "regression",
        "paper_target": "methane-hydrate saturation (Smh)",
        "reference_method": "Archie electrical-resistivity method",
        "ground_truth_basis": "Archie-derived Smh from high-resolution resistivity interpretation",
        "target_formula_inputs": "Rt; porosity; Archie parameters and water resistivity",
        "model_inputs_reported": "rho; density porosity; Rt; GR; Vp",
        "audit_implication": "Rt/porosity predictor overlap can reconstruct the target equation and must be reported as a circularity sensitivity.",
    },
    {
        "paper": "Chong et al. (2024)",
        "task": "classification",
        "paper_target": "occurrence: none / fracture-filling / pore-filling",
        "reference_method": "RAB-image and multi-log interpretation",
        "ground_truth_basis": "point-by-point RAB occurrence assignment; resistivity and Vp supported interpretation",
        "target_formula_inputs": "not a saturation threshold",
        "model_inputs_reported": "rho; density porosity; Rt; GR; Vp",
        "audit_implication": "Occurrence labels must be independent records; never manufacture them from Sh cutoffs.",
    },
]

print("Data directory:", DATA_DIR)
print("Run mode:", RUN_MODE)
print("Fixed blind-well split:", TRAIN_WELLS, "->", BLIND_WELL)


## 1. Workbook parsing and target discovery

The parser supports both the stacked Mallik layout (role, mnemonic, unit, description, data) and the direct MTE/IGS layouts. It inventories every source column before deciding whether it is a feature, context field, or target candidate.

Target discovery is intentionally broad. A column is listed when its role says `ground truth`, `target`, `label`, or `response`, or when its header clearly resembles hydrate saturation, occurrence, or water/residual saturation. Discovery does not authorize modeling.

In [ ]:
# =============================================================================
# Parsing utilities
# =============================================================================

def normalize_token(value: Any) -> str:
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass
    return "".join(ch for ch in str(value).strip().lower() if ch.isalnum())


def clean_label(value: Any, fallback: str = "") -> str:
    try:
        missing = pd.isna(value)
    except Exception:
        missing = False
    text = "" if missing else str(value).strip()
    text = re.sub(r"\s+", " ", text)
    return text or fallback


def safe_slug(value: Any, fallback: str = "column") -> str:
    text = re.sub(r"[^A-Za-z0-9]+", "_", clean_label(value, fallback)).strip("_").lower()
    return text or fallback


def dedupe_labels(labels: Iterable[str]) -> list[str]:
    counts: dict[str, int] = {}
    result: list[str] = []
    for label in labels:
        count = counts.get(label, 0)
        counts[label] = count + 1
        result.append(label if count == 0 else f"{label}__{count + 1}")
    return result


def numeric_series(values: pd.Series) -> pd.Series:
    cleaned = (
        values.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan, "-": np.nan, "--": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce").replace([np.inf, -np.inf], np.nan)


FEATURE_ALIASES: dict[str, tuple[str, ...]] = {
    "measured_depth": (
        "depth", "depth_ft", "depth, ft", "depth ft", "depth m", "depth_m", "dept",
        "measured depth", "md",
    ),
    "true_vertical_depth": ("true vertical depth", "tvd", "tvdss", "true depth"),
    "rhob_g_cc": (
        "rho_b", "rhob", "density_gpcc", "density_gcpcc", "density g/cc", "density gpcc",
        "bulk density", "density",
    ),
    "density_porosity_vv": (
        "phi_porosity", "dphi", "phi_den", "density porosity", "density_porosity", "phid",
    ),
    "neutron_porosity_vv": ("nphi", "phi_neut", "neutron porosity", "phin"),
    "nmr_porosity_vv": ("nmrphi", "phi_nmr", "nmr porosity", "nmr_phi"),
    "gr_api": ("gr", "gamma ray", "gamma_ray", "gamma radiation"),
    "caliper_in": ("caliper", "cal1", "cali", "cal"),
    "differential_caliper_mm": ("differential caliper", "differential_caliper", "diff caliper"),
    "density_correction_g_cc": ("drho", "density correction", "rhoc", "delta rho"),
    "rt_ohm_m": (
        "res", "rt", "deep formation resistivity", "apparent resistivity", "deep resistivity",
        "resistivity", "p40h",
    ),
    "vp_m_s": ("vp", "velp", "compressional wave velocity", "compressional velocity", "p wave velocity"),
    "vs_m_s": ("vs", "vs1", "shear wave velocity", "shear velocity", "s wave velocity"),
}

FEATURE_ALIAS_LOOKUP: dict[str, str] = {}
for canonical, aliases in FEATURE_ALIASES.items():
    for alias in (canonical, *aliases):
        FEATURE_ALIAS_LOOKUP[normalize_token(alias)] = canonical

ROLE_TARGET_TOKENS = {
    "target", "groundtruth", "groundtruthvalue", "label", "response", "mloutput", "output",
    "classificationtarget", "regressiontarget",
}
ROLE_FEATURE_TOKENS = {"mlinput", "feature", "predictor", "input"}
ROLE_TOKENS = ROLE_TARGET_TOKENS | ROLE_FEATURE_TOKENS | {"depth", "qc", "outlierremoval", "context"}

EXACT_TARGET_HEADERS = {
    "sgh", "sh", "smh", "nmrsat", "hydratesaturation", "gashydratesaturation",
    "methanehydratesaturation", "swr", "swirr", "watersaturation", "residualwatersaturation",
    "hydrateoccurrence", "occurrence", "hydratetype", "occurrenceclass",
}


def feature_for_header(header: Any) -> str | None:
    token = normalize_token(header)
    # A090/AF90 are not mapped automatically because their physical meaning must be verified.
    if token in {"a090", "af90"}:
        return None
    return FEATURE_ALIAS_LOOKUP.get(token)


def header_row_score(row: pd.Series) -> tuple[int, int, int]:
    tokens = [normalize_token(value) for value in row.tolist() if normalize_token(value)]
    exact_features = sum(token in FEATURE_ALIAS_LOOKUP for token in tokens)
    exact_targets = sum(token in EXACT_TARGET_HEADERS for token in tokens)
    keywords = sum(
        any(part in token for part in (
            "depth", "density", "porosity", "caliper", "resist", "gamma", "velocity",
            "saturation", "hydrate", "occurrence",
        ))
        for token in tokens
    )
    return exact_features * 10 + exact_targets * 5 + keywords, exact_features + exact_targets, len(tokens)


def detect_header_layout(raw: pd.DataFrame, max_scan_rows: int = 12) -> dict[str, Any]:
    if raw.empty:
        raise ValueError("Sheet is empty.")
    scan_count = min(max_scan_rows, len(raw))
    scores = [header_row_score(raw.iloc[index]) for index in range(scan_count)]
    header_index = max(range(scan_count), key=lambda index: scores[index])
    if scores[header_index][1] < 3:
        raise ValueError(f"Could not identify a reliable mnemonic row. Scores: {scores}")

    header_values = [clean_label(value, f"unnamed_{i}") for i, value in enumerate(raw.iloc[header_index])]
    meaningful = [i for i, value in enumerate(header_values) if not value.startswith("unnamed_")]

    data_start = None
    for row_index in range(header_index + 1, min(len(raw), header_index + 15)):
        row = raw.iloc[row_index, meaningful] if meaningful else raw.iloc[row_index]
        nonempty = int(row.notna().sum())
        numeric = int(numeric_series(row).notna().sum())
        if numeric >= 3 and numeric / max(nonempty, 1) >= 0.40:
            data_start = row_index
            break
    if data_start is None:
        raise ValueError("Could not identify the first numeric data row after the mnemonic row.")

    role_index = None
    for row_index in range(max(0, header_index - 2), header_index):
        row_tokens = {normalize_token(value) for value in raw.iloc[row_index].tolist()}
        if row_tokens & ROLE_TOKENS:
            role_index = row_index

    unit_index = None
    description_index = None
    for row_index in range(header_index + 1, data_start):
        tokens = [normalize_token(value) for value in raw.iloc[row_index].tolist() if normalize_token(value)]
        unit_hits = sum(
            token in {"m", "ft", "kgm3", "gcc", "gcm3", "api", "mm", "in", "ohmm", "ms", "kms", "fraction", "percent"}
            or any(unit in token for unit in ("kgm3", "ohmm", "api", "inch", "feet", "percent"))
            for token in tokens
        )
        if unit_hits >= 2 and unit_index is None:
            unit_index = row_index
        description_index = row_index

    return {
        "header_index": header_index,
        "data_start": data_start,
        "role_index": role_index,
        "unit_index": unit_index,
        "description_index": description_index,
        "header_score": scores[header_index][0],
        "recognized_headers": scores[header_index][1],
    }


def choose_best_sheet(path: Path) -> tuple[str, pd.DataFrame, dict[str, Any]]:
    with pd.ExcelFile(path) as excel:
        sheet_names = list(excel.sheet_names)
    candidates: list[tuple[int, int, str, pd.DataFrame, dict[str, Any]]] = []
    errors: list[str] = []
    for sheet_name in sheet_names:
        if "refined" in normalize_token(sheet_name):
            continue
        try:
            raw = pd.read_excel(path, sheet_name=sheet_name, header=None)
            layout = detect_header_layout(raw)
            candidates.append((layout["recognized_headers"], len(raw), sheet_name, raw, layout))
        except Exception as exc:
            errors.append(f"{sheet_name}: {exc}")
    if not candidates:
        raise ValueError(f"No model-input sheet found in {path.name}. Details: {errors}")
    _, _, sheet_name, raw, layout = sorted(candidates, key=lambda item: (item[0], item[1]), reverse=True)[0]
    return sheet_name, raw, layout


def read_source_sheet(path: Path, requested_sheet: str | None) -> tuple[str, pd.DataFrame, dict[str, Any]]:
    if requested_sheet is None:
        return choose_best_sheet(path)
    with pd.ExcelFile(path) as excel:
        matching = {normalize_token(name): name for name in excel.sheet_names}
    actual = matching.get(normalize_token(requested_sheet))
    if actual is None:
        raise ValueError(f"Sheet {requested_sheet!r} is missing from {path.name}.")
    raw = pd.read_excel(path, sheet_name=actual, header=None)
    return actual, raw, detect_header_layout(raw)


def parse_source(spec: dict[str, Any], data_dir: Path) -> dict[str, Any]:
    path = data_dir / spec["filename"]
    if not path.exists():
        raise FileNotFoundError(path)
    sheet_name, raw, layout = read_source_sheet(path, spec.get("sheet_name"))

    headers = dedupe_labels([
        clean_label(value, f"unnamed_{i}")
        for i, value in enumerate(raw.iloc[layout["header_index"]].tolist())
    ])
    roles = (
        [clean_label(value) for value in raw.iloc[layout["role_index"]].tolist()]
        if layout["role_index"] is not None else [""] * len(headers)
    )
    units = (
        [clean_label(value) for value in raw.iloc[layout["unit_index"]].tolist()]
        if layout["unit_index"] is not None else [""] * len(headers)
    )
    descriptions = (
        [clean_label(value) for value in raw.iloc[layout["description_index"]].tolist()]
        if layout["description_index"] is not None else [""] * len(headers)
    )

    data = raw.iloc[layout["data_start"]:].copy().reset_index(drop=True)
    data.columns = headers
    data = data.dropna(axis=0, how="all").dropna(axis=1, how="all")

    metadata_rows: list[dict[str, Any]] = []
    for position, header in enumerate(headers):
        if header not in data.columns:
            continue
        metadata_rows.append({
            "well_alias": spec["well_alias"],
            "source_label": spec["source_label"],
            "source_workbook": spec["filename"],
            "source_sheet": sheet_name,
            "column_position": position + 1,
            "source_header": header,
            "source_role": roles[position] if position < len(roles) else "",
            "source_unit": units[position] if position < len(units) else "",
            "source_description": descriptions[position] if position < len(descriptions) else "",
            "feature_mapping": feature_for_header(header) or "unmapped",
            "non_null_rows": int(data[header].notna().sum()),
        })

    return {
        "spec": spec,
        "path": path,
        "sheet_name": sheet_name,
        "raw": raw,
        "layout": layout,
        "data": data,
        "metadata": pd.DataFrame(metadata_rows),
    }


## 2. Target semantics and target-contract audit

The target catalog distinguishes **notation** from **derivation**:

- `NMR_SAT` suggests an NMR-density-derived hydrate saturation but still requires source confirmation.
- `Sgh`, `S_h`, `Sh`, and `Smh` are generic hydrate-saturation notation; the name alone does not identify whether the values came from NMR-density, Archie resistivity, acoustic velocity, manual interpretation, or a previous model.
- `S_wr`, `Swr`, and related names are water/residual saturation candidates and are excluded from this project’s ML targets.
- hydrate occurrence is categorical and must be independently interpreted. It may not be manufactured by thresholding a saturation column.

When two numeric target candidates occur in the same well, the notebook computes overlap, correlation, and direct differences. Similar distributions are not enough to prove equivalence; formulas and provenance still control approval.

In [ ]:
# =============================================================================
# Target discovery and semantic audit
# =============================================================================

def _role_is_target(role: Any) -> bool:
    token = normalize_token(role)
    return token in ROLE_TARGET_TOKENS or any(part in token for part in ("groundtruth", "target", "response", "label"))


def infer_target_semantics(
    header: str,
    role: str,
    description: str,
    values: pd.Series,
) -> dict[str, Any]:
    header_token = normalize_token(header)
    text_token = normalize_token(f"{header} {role} {description}")
    numeric = numeric_series(values)
    numeric_fraction = float(numeric.notna().mean()) if len(values) else 0.0
    raw_unique = values.dropna().astype(str).str.strip().nunique()
    numeric_unique = numeric.dropna().nunique()

    is_candidate = _role_is_target(role)
    is_candidate = is_candidate or header_token in EXACT_TARGET_HEADERS
    is_candidate = is_candidate or (
        "hydrate" in text_token and any(part in text_token for part in ("sat", "occurrence", "class", "type"))
    )
    is_candidate = is_candidate or header_token in {"sgh", "sh", "smh", "swr", "swirr", "nmrsat"}

    family = "other_or_unknown_target"
    derivation = "unknown"
    default_decision = "manual_review"
    exclusion_reason = ""

    if any(part in text_token for part in ("occurrence", "hydrateclass", "hydratetype", "porefilling", "fracturefilling")):
        family = "hydrate_occurrence"
        derivation = "independent_interpretation_not_yet_verified"
    elif header_token in {"swr", "swirr", "watersaturation", "residualwatersaturation"} or (
        "water" in text_token and "saturation" in text_token
    ):
        family = "water_or_residual_saturation"
        derivation = "water_saturation_definition_not_verified"
        default_decision = "excluded_from_project_targets"
        exclusion_reason = "Project outputs are hydrate occurrence and hydrate saturation, not water saturation."
    elif header_token == "nmrsat" or ("nmr" in text_token and "sat" in text_token):
        family = "hydrate_saturation"
        derivation = "nmr_density_likely_but_source_confirmation_required"
    elif header_token in {"sgh", "sh", "smh", "hydratesaturation", "gashydratesaturation", "methanehydratesaturation"} or (
        "hydrate" in text_token and "saturation" in text_token
    ):
        family = "hydrate_saturation"
        derivation = "generic_notation_derivation_unknown"

    if family == "hydrate_occurrence":
        task = "classification"
    elif numeric_fraction >= 0.80 and numeric_unique > 10:
        task = "regression"
    elif raw_unique <= 20:
        task = "classification"
    else:
        task = "unknown"

    return {
        "is_target_candidate": bool(is_candidate),
        "target_family": family,
        "inferred_task": task,
        "inferred_derivation": derivation,
        "default_decision": default_decision,
        "exclusion_reason": exclusion_reason,
        "numeric_fraction": numeric_fraction,
        "raw_unique_values": int(raw_unique),
        "numeric_unique_values": int(numeric_unique),
    }


def build_target_catalog(parsed_sources: dict[str, dict[str, Any]]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for well_alias, parsed in parsed_sources.items():
        data = parsed["data"]
        for meta in parsed["metadata"].to_dict("records"):
            header = meta["source_header"]
            semantics = infer_target_semantics(
                header,
                meta["source_role"],
                meta["source_description"],
                data[header],
            )
            if not semantics["is_target_candidate"]:
                continue
            raw_key = f"raw_target__{meta['column_position']:03d}__{safe_slug(header)}"
            numeric = numeric_series(data[header])
            finite = numeric.dropna()
            rows.append({
                **meta,
                **semantics,
                "raw_target_key": raw_key,
                "numeric_rows": int(finite.size),
                "numeric_min": float(finite.min()) if not finite.empty else np.nan,
                "numeric_max": float(finite.max()) if not finite.empty else np.nan,
                "numeric_mean": float(finite.mean()) if not finite.empty else np.nan,
                "looks_percent_scaled": bool(not finite.empty and finite.quantile(0.99) > 1.5 and finite.quantile(0.99) <= 100.5),
            })
    return pd.DataFrame(rows)


def _normalize_numeric_target_for_comparison(values: pd.Series) -> tuple[pd.Series, str]:
    numeric = numeric_series(values)
    finite = numeric.dropna()
    if finite.empty:
        return numeric, "no_data"
    q99 = float(finite.quantile(0.99))
    if q99 > 1.5 and q99 <= 100.5:
        return numeric / 100.0, "percent_to_fraction"
    return numeric, "identity"


def build_target_pair_checks(
    parsed_sources: dict[str, dict[str, Any]],
    target_catalog: pd.DataFrame,
) -> pd.DataFrame:
    """Compare only candidates that claim the same scientific target family.

    Cross-family comparisons (for example hydrate saturation versus residual
    water saturation) are intentionally omitted because they do not establish
    target equivalence and make the audit harder to interpret.
    """
    rows: list[dict[str, Any]] = []
    if target_catalog.empty:
        return pd.DataFrame(rows)
    for well_alias, well_group in target_catalog.groupby("well_alias"):
        data = parsed_sources[well_alias]["data"]
        for family, family_group in well_group.groupby("target_family"):
            records = family_group.to_dict("records")
            if len(records) < 2:
                continue
            for left, right in combinations(records, 2):
                if family == "hydrate_occurrence":
                    pair = pd.DataFrame({
                        "left": data[left["source_header"]].map(normalize_token),
                        "right": data[right["source_header"]].map(normalize_token),
                    })
                    pair = pair[(pair["left"] != "") & (pair["right"] != "")]
                    rows.append({
                        "well_alias": well_alias,
                        "target_family": family,
                        "left_header": left["source_header"],
                        "right_header": right["source_header"],
                        "comparison_type": "categorical_label_agreement",
                        "left_scaling": "categorical",
                        "right_scaling": "categorical",
                        "overlap_rows": int(len(pair)),
                        "pearson_correlation": np.nan,
                        "mae_normalized": np.nan,
                        "rmse_normalized": np.nan,
                        "max_absolute_difference": np.nan,
                        "fraction_equal_within_0_001": np.nan,
                        "fraction_exact_label_match": float((pair["left"] == pair["right"]).mean()) if len(pair) else np.nan,
                        "equivalence_status": "requires_label_definition_and_provenance_review",
                    })
                    continue

                left_values, left_scaling = _normalize_numeric_target_for_comparison(data[left["source_header"]])
                right_values, right_scaling = _normalize_numeric_target_for_comparison(data[right["source_header"]])
                pair = pd.DataFrame({"left": left_values, "right": right_values}).dropna()
                if pair.empty:
                    corr = mae = rmse = max_abs = within_1e3 = np.nan
                else:
                    diff = pair["left"] - pair["right"]
                    corr = float(pair.corr().iloc[0, 1]) if len(pair) >= 3 and pair.nunique().min() > 1 else np.nan
                    mae = float(np.abs(diff).mean())
                    rmse = float(np.sqrt(np.mean(diff**2)))
                    max_abs = float(np.abs(diff).max())
                    within_1e3 = float((np.abs(diff) <= 1e-3).mean())
                rows.append({
                    "well_alias": well_alias,
                    "target_family": family,
                    "left_header": left["source_header"],
                    "right_header": right["source_header"],
                    "comparison_type": "numeric_after_unit_scaling",
                    "left_scaling": left_scaling,
                    "right_scaling": right_scaling,
                    "overlap_rows": int(len(pair)),
                    "pearson_correlation": corr,
                    "mae_normalized": mae,
                    "rmse_normalized": rmse,
                    "max_absolute_difference": max_abs,
                    "fraction_equal_within_0_001": within_1e3,
                    "fraction_exact_label_match": np.nan,
                    "equivalence_status": "requires_formula_and_provenance_review",
                })
    return pd.DataFrame(rows)


def suggest_target_headers(target_catalog: pd.DataFrame) -> pd.DataFrame:
    if target_catalog.empty:
        return pd.DataFrame()
    priority = {
        "nmr_density_likely_but_source_confirmation_required": 0,
        "generic_notation_derivation_unknown": 1,
        "independent_interpretation_not_yet_verified": 2,
        "unknown": 9,
    }
    suggestions = target_catalog.copy()
    suggestions["suggestion_priority"] = suggestions["inferred_derivation"].map(priority).fillna(8)
    suggestions["decision_required"] = np.where(
        suggestions["default_decision"].eq("excluded_from_project_targets"),
        "retain_for_audit_only",
        "verify_equation_and_provenance_before_contract",
    )
    columns = [
        "well_alias", "target_family", "source_header", "source_role", "source_unit",
        "source_description", "inferred_task", "inferred_derivation", "numeric_rows",
        "numeric_min", "numeric_max", "looks_percent_scaled", "default_decision",
        "exclusion_reason", "suggestion_priority", "decision_required",
    ]
    return suggestions.sort_values(
        ["well_alias", "target_family", "suggestion_priority", "numeric_rows"],
        ascending=[True, True, True, False],
    ).reindex(columns=columns).reset_index(drop=True)


def paper_target_methods_table() -> pd.DataFrame:
    return pd.DataFrame(PAPER_TARGET_METHODS)


def current_contract_table() -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for target_name, config in TARGET_CONTRACTS.items():
        for well_alias, source_header in config["source_header_by_well"].items():
            rows.append({
                "model_target": target_name,
                "task": config["task"],
                "enabled": config["enabled"],
                "approved": config["approved"],
                "well_alias": well_alias,
                "selected_source_header": source_header,
                "required_provenance": config["required_provenance"],
                "reference_method": config.get("reference_method"),
                "target_definition": config.get("target_definition"),
                "source_document": config.get("source_document"),
                "independent_from_saturation": config.get("independent_from_saturation", np.nan),
                "formula_input_features": ", ".join(config.get("formula_input_features", [])),
                "labeling_evidence_features": ", ".join(config.get("labeling_evidence_features", [])),
            })
    return pd.DataFrame(rows)


## 3. Feature standardization and provenance

Feature aliases may be standardized because they refer to measurements with known physical units. Target aliases are never standardized this way.

Measured depth and true vertical depth remain separate. Depth is retained for plotting and alignment but is not used as a predictor. When multiple source columns map to one feature, the notebook selects deterministically from role, exact mnemonic match, and coverage, and records every candidate in the feature-mapping audit.

In [ ]:
# =============================================================================
# Feature standardization
# =============================================================================

def infer_unit(header: str, unit: str, default: str, canonical: str) -> str:
    combined = f"{header} {unit}".lower().replace("³", "3")
    token = normalize_token(combined)
    if canonical in {"measured_depth", "true_vertical_depth"}:
        if "ft" in combined or "feet" in combined:
            return "ft"
        if re.search(r"(^|\s)m($|\s)", combined) and "mm" not in combined and "/s" not in combined:
            return "m"
    if canonical == "rhob_g_cc":
        if "kg/m3" in combined or "kgm3" in token:
            return "kg/m3"
        if any(value in combined for value in ("g/cc", "g/cm3", "gpcc", "gcpcc")):
            return "g/cc"
    if canonical in {"caliper_in", "differential_caliper_mm"}:
        if "mm" in combined:
            return "mm"
        if any(value in combined for value in ("inch", " in", "in.")):
            return "in"
    if canonical in {"vp_m_s", "vs_m_s"}:
        if "km/s" in combined or "kms" in token:
            return "km/s"
        if "m/s" in combined or token.endswith("ms"):
            return "m/s"
    if canonical in {"density_porosity_vv", "neutron_porosity_vv", "nmr_porosity_vv"}:
        if "%" in combined or "percent" in combined:
            return "percent"
        return "fraction_or_unknown"
    return default


def convert_fraction(values: pd.Series) -> tuple[pd.Series, str]:
    numeric = numeric_series(values)
    finite = numeric.dropna()
    if finite.empty:
        return numeric, "no_data"
    q99 = float(finite.quantile(0.99))
    if q99 > 1.5 and q99 <= 100.5:
        return numeric / 100.0, "percent_to_fraction"
    return numeric, "identity"


def _feature_candidate_score(meta: dict[str, Any], data: pd.DataFrame, canonical: str) -> tuple[int, int, int]:
    header_token = normalize_token(meta["source_header"])
    role_token = normalize_token(meta["source_role"])
    exact = int(header_token == normalize_token(canonical))
    role_bonus = int(role_token in ROLE_FEATURE_TOKENS or "mlinput" in role_token)
    coverage = int(data[meta["source_header"]].notna().sum())
    return role_bonus, exact, coverage


def standardize_features_and_targets(
    parsed: dict[str, Any],
    target_catalog: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    spec = parsed["spec"]
    data = parsed["data"]
    metadata = parsed["metadata"].copy()
    well_alias = spec["well_alias"]

    standardized = pd.DataFrame(index=data.index)
    standardized["well_alias"] = well_alias
    standardized["source_label"] = spec["source_label"]
    standardized["source_workbook"] = spec["filename"]
    standardized["source_sheet"] = parsed["sheet_name"]
    standardized["source_row"] = np.arange(parsed["layout"]["data_start"] + 1, parsed["layout"]["data_start"] + 1 + len(data))

    mapping_rows: list[dict[str, Any]] = []
    unit_rows: list[dict[str, Any]] = []

    candidates_by_feature: dict[str, list[dict[str, Any]]] = {}
    for meta in metadata.to_dict("records"):
        canonical = feature_for_header(meta["source_header"])
        if canonical:
            candidates_by_feature.setdefault(canonical, []).append(meta)

    for canonical, candidates in candidates_by_feature.items():
        selected = sorted(
            candidates,
            key=lambda meta: _feature_candidate_score(meta, data, canonical),
            reverse=True,
        )[0]
        for meta in candidates:
            mapping_rows.append({
                **meta,
                "canonical_feature": canonical,
                "selected_for_feature": meta["source_header"] == selected["source_header"],
                "selection_score": str(_feature_candidate_score(meta, data, canonical)),
            })

        header = selected["source_header"]
        values = numeric_series(data[header])
        unit_text = selected["source_unit"]

        if canonical in {"measured_depth", "true_vertical_depth"}:
            unit = infer_unit(header, unit_text, spec["default_depth_unit"], canonical)
            standardized[f"{canonical}_original"] = values
            standardized[f"{canonical}_original_unit"] = unit
            standardized[f"{canonical}_m"] = values * 0.3048 if unit == "ft" else values
            unit_rows.append({
                "well_alias": well_alias, "field": f"{canonical}_m", "source_header": header,
                "source_unit": unit, "canonical_unit": "m",
                "conversion": "multiply_0.3048" if unit == "ft" else "identity",
            })
        elif canonical == "rhob_g_cc":
            unit = infer_unit(header, unit_text, spec["default_density_unit"], canonical)
            finite = values.dropna()
            if unit == "g/cc" and not finite.empty and float(finite.median()) > 20:
                unit = "kg/m3"
            standardized[canonical] = values / 1000.0 if unit == "kg/m3" else values
            unit_rows.append({
                "well_alias": well_alias, "field": canonical, "source_header": header,
                "source_unit": unit, "canonical_unit": "g/cc",
                "conversion": "divide_1000" if unit == "kg/m3" else "identity",
            })
        elif canonical in {"density_porosity_vv", "neutron_porosity_vv", "nmr_porosity_vv"}:
            converted, conversion = convert_fraction(values)
            standardized[canonical] = converted
            unit_rows.append({
                "well_alias": well_alias, "field": canonical, "source_header": header,
                "source_unit": infer_unit(header, unit_text, "fraction_or_unknown", canonical),
                "canonical_unit": "fraction", "conversion": conversion,
            })
        elif canonical == "caliper_in":
            unit = infer_unit(header, unit_text, spec["default_caliper_unit"], canonical)
            finite = values.dropna()
            if unit == "in" and not finite.empty and float(finite.median()) > 50:
                unit = "mm"
            standardized[canonical] = values / 25.4 if unit == "mm" else values
            unit_rows.append({
                "well_alias": well_alias, "field": canonical, "source_header": header,
                "source_unit": unit, "canonical_unit": "in",
                "conversion": "divide_25.4" if unit == "mm" else "identity",
            })
        elif canonical == "differential_caliper_mm":
            unit = infer_unit(header, unit_text, "mm", canonical)
            standardized[canonical] = values * 25.4 if unit == "in" else values
            unit_rows.append({
                "well_alias": well_alias, "field": canonical, "source_header": header,
                "source_unit": unit, "canonical_unit": "mm",
                "conversion": "multiply_25.4" if unit == "in" else "identity",
            })
        elif canonical in {"vp_m_s", "vs_m_s"}:
            unit = infer_unit(header, unit_text, "m/s", canonical)
            finite = values.dropna()
            if unit == "m/s" and not finite.empty and float(finite.median()) < 20:
                unit = "km/s"
            standardized[canonical] = values * 1000.0 if unit == "km/s" else values
            unit_rows.append({
                "well_alias": well_alias, "field": canonical, "source_header": header,
                "source_unit": unit, "canonical_unit": "m/s",
                "conversion": "multiply_1000" if unit == "km/s" else "identity",
            })
        else:
            standardized[canonical] = values
            unit_rows.append({
                "well_alias": well_alias, "field": canonical, "source_header": header,
                "source_unit": unit_text or "unknown", "canonical_unit": "as_source",
                "conversion": "identity",
            })

    if "measured_depth_m" not in standardized:
        if "true_vertical_depth_m" in standardized:
            standardized["measured_depth_m"] = standardized["true_vertical_depth_m"]
            unit_rows.append({
                "well_alias": well_alias, "field": "measured_depth_m", "source_header": "fallback_true_vertical_depth",
                "source_unit": "m", "canonical_unit": "m", "conversion": "fallback_copy",
            })
        else:
            raise ValueError(f"No measured or true-vertical depth mapped for {well_alias}.")

    # Retain each raw target candidate as a separate column.
    well_targets = target_catalog[target_catalog["well_alias"] == well_alias] if not target_catalog.empty else pd.DataFrame()
    for target in well_targets.to_dict("records"):
        standardized[target["raw_target_key"]] = data[target["source_header"]]

    # Derived predictors stay transparent and optional.
    if "rt_ohm_m" in standardized:
        rt = pd.to_numeric(standardized["rt_ohm_m"], errors="coerce")
        standardized["log10_rt_ohm_m"] = np.log10(rt.where(rt > 0))

    numeric_columns = [
        column for column in standardized.columns
        if column not in {
            "well_alias", "source_label", "source_workbook", "source_sheet",
            "measured_depth_original_unit", "true_vertical_depth_original_unit",
        } and not column.startswith("raw_target__")
    ]
    for column in numeric_columns:
        standardized[column] = pd.to_numeric(standardized[column], errors="coerce").replace([np.inf, -np.inf], np.nan)

    keep = standardized["measured_depth_m"].notna()
    signal_columns = [column for column in standardized.columns if column in FEATURE_ALIAS_LOOKUP.values() or column.startswith("raw_target__")]
    if signal_columns:
        keep &= standardized[signal_columns].notna().any(axis=1)
    standardized = standardized.loc[keep].sort_values(["measured_depth_m", "source_row"], kind="stable").reset_index(drop=True)

    return standardized, pd.DataFrame(mapping_rows), pd.DataFrame(unit_rows)


def build_readiness_audit(well_frames: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    all_features = sorted(set(sum((list(panel) for panel in FEATURE_PANELS.values()), [])))
    for well_alias, frame in well_frames.items():
        depth = frame["measured_depth_m"].dropna()
        row = {
            "well_alias": well_alias,
            "rows": len(frame),
            "depth_min_m": float(depth.min()) if not depth.empty else np.nan,
            "depth_max_m": float(depth.max()) if not depth.empty else np.nan,
            "depth_span_m": float(depth.max() - depth.min()) if not depth.empty else np.nan,
            "depth_monotonic": bool(depth.is_monotonic_increasing),
            "duplicate_depth_rows": int(frame["measured_depth_m"].duplicated().sum()),
        }
        for feature in all_features:
            row[f"coverage__{feature}"] = float(frame[feature].notna().mean()) if feature in frame else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


## 4. Preflight audit

This cell reads the three original workbooks, builds four neutral well tables, inventories every target-like column, and writes a single audit workbook. It does not train a model in `audit` mode.

The most important review questions are:

1. Which exact source header is the reference hydrate saturation in each well?
2. What equation or interpretation produced it?
3. Are generic `Sh`/`Sgh` columns equivalent to `NMR_SAT`, or are they outputs from a different method?
4. Is an occurrence label independently interpreted, and does it contain the three expected classes?
5. Which features participate in the approved saturation formula and therefore require a circularity sensitivity check?

In [ ]:
# =============================================================================
# Load and audit all sources
# =============================================================================

def load_and_audit_sources(data_dir: Path = DATA_DIR) -> dict[str, Any]:
    missing = [spec["filename"] for spec in SOURCE_SPECS if not (data_dir / spec["filename"]).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing original workbooks in " + str(data_dir) + ": " + ", ".join(sorted(set(missing)))
        )

    parsed_sources = {spec["well_alias"]: parse_source(spec, data_dir) for spec in SOURCE_SPECS}
    target_catalog = build_target_catalog(parsed_sources)
    target_pair_checks = build_target_pair_checks(parsed_sources, target_catalog)
    target_suggestions = suggest_target_headers(target_catalog)

    well_frames: dict[str, pd.DataFrame] = {}
    feature_mappings: list[pd.DataFrame] = []
    unit_audits: list[pd.DataFrame] = []
    layouts: list[dict[str, Any]] = []
    source_columns: list[pd.DataFrame] = []

    for well_alias, parsed in parsed_sources.items():
        frame, mapping, units = standardize_features_and_targets(parsed, target_catalog)
        well_frames[well_alias] = frame
        feature_mappings.append(mapping)
        unit_audits.append(units)
        source_columns.append(parsed["metadata"])
        layout = parsed["layout"]
        layouts.append({
            "well_alias": well_alias,
            "source_workbook": parsed["spec"]["filename"],
            "source_sheet": parsed["sheet_name"],
            "header_row_excel": layout["header_index"] + 1,
            "data_start_row_excel": layout["data_start"] + 1,
            "role_row_excel": layout["role_index"] + 1 if layout["role_index"] is not None else None,
            "unit_row_excel": layout["unit_index"] + 1 if layout["unit_index"] is not None else None,
            "description_row_excel": layout["description_index"] + 1 if layout["description_index"] is not None else None,
            "recognized_header_count": layout["recognized_headers"],
            "standardized_rows": len(frame),
        })

    return {
        "parsed_sources": parsed_sources,
        "well_frames": well_frames,
        "source_columns": pd.concat(source_columns, ignore_index=True, sort=False),
        "source_layout": pd.DataFrame(layouts),
        "target_catalog": target_catalog,
        "target_pair_checks": target_pair_checks,
        "target_suggestions": target_suggestions,
        "target_contract": current_contract_table(),
        "paper_target_methods": paper_target_methods_table(),
        "feature_mapping": pd.concat(feature_mappings, ignore_index=True, sort=False) if feature_mappings else pd.DataFrame(),
        "unit_audit": pd.concat(unit_audits, ignore_index=True, sort=False) if unit_audits else pd.DataFrame(),
        "well_readiness": build_readiness_audit(well_frames),
    }


def prepare_known_output_paths() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    for path in (RESULTS_XLSX, PREDICTIONS_PARQUET, FIGURES_PDF, SELECTED_MODELS):
        if path.exists():
            path.unlink()


def write_audit_workbook(audit: dict[str, Any]) -> None:
    prepare_known_output_paths()
    summary = pd.DataFrame([
        {"item": "run_mode", "value": RUN_MODE},
        {"item": "created_utc", "value": datetime.now(timezone.utc).isoformat()},
        {"item": "data_directory", "value": str(DATA_DIR)},
        {"item": "train_wells", "value": ", ".join(TRAIN_WELLS)},
        {"item": "blind_well", "value": BLIND_WELL},
        {"item": "modeling_status", "value": "blocked_until_explicit_target_contract_approval"},
    ])
    with pd.ExcelWriter(RESULTS_XLSX, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="run_summary", index=False)
        audit["target_catalog"].to_excel(writer, sheet_name="target_catalog", index=False)
        audit["target_pair_checks"].to_excel(writer, sheet_name="target_pair_checks", index=False)
        audit["target_suggestions"].to_excel(writer, sheet_name="target_suggestions", index=False)
        audit["target_contract"].to_excel(writer, sheet_name="target_contract", index=False)
        audit["paper_target_methods"].to_excel(writer, sheet_name="paper_target_methods", index=False)
        audit["source_layout"].to_excel(writer, sheet_name="source_layout", index=False)
        audit["source_columns"].to_excel(writer, sheet_name="source_columns", index=False)
        audit["feature_mapping"].to_excel(writer, sheet_name="feature_mapping", index=False)
        audit["unit_audit"].to_excel(writer, sheet_name="unit_audit", index=False)
        audit["well_readiness"].to_excel(writer, sheet_name="well_readiness", index=False)


audit = load_and_audit_sources()
write_audit_workbook(audit)

print("Audit workbook:", RESULTS_XLSX)
print("\nTARGET CATALOG")
display(audit["target_catalog"])
print("\nPAIRWISE TARGET CHECKS")
display(audit["target_pair_checks"])
print("\nTARGET HEADER SUGGESTIONS — suggestions are not approvals")
display(audit["target_suggestions"])
print("\nSOURCE-PAPER TARGET METHODS")
display(audit["paper_target_methods"])
print("\nCURRENT TARGET CONTRACT")
display(audit["target_contract"])


## 5. Contract resolution and modeling tables

After the audit:

1. return to the configuration cell;
2. enter one exact `source_header` per approved well;
3. set the target contract’s `approved` field to `True`;
4. document direct formula inputs in `formula_input_features`;
5. enable occurrence only when its labels are independent from saturation;
6. change `RUN_MODE` to `model`.

The resolver hard-stops on missing headers, wrong target families, unsupported occurrence classes, out-of-range saturation, or an unapproved contract.

In [ ]:
# =============================================================================
# Target contract resolution
# =============================================================================

def _catalog_match(target_catalog: pd.DataFrame, well_alias: str, source_header: str) -> pd.Series:
    matches = target_catalog[
        (target_catalog["well_alias"] == well_alias)
        & (target_catalog["source_header"].map(normalize_token) == normalize_token(source_header))
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one target-catalog match for {well_alias}/{source_header!r}; found {len(matches)}. "
            "Use the exact source_header shown in target_catalog."
        )
    return matches.iloc[0]


def normalize_occurrence_label(value: Any) -> str | float:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    token = normalize_token(value)
    if not token:
        return np.nan
    if token in {"0", "none", "nohydrate", "nonhydrate", "absent", "background"} or "nohydrate" in token:
        return "none"
    if token in {"1", "fracture", "fracturefill", "fracturefilling"} or "fracture" in token:
        return "fracture"
    if token in {"2", "pore", "porefill", "porefilling"} or "pore" in token:
        return "pore"
    return f"unrecognized::{clean_label(value)}"


def resolve_target_contracts(
    audit: dict[str, Any],
) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    if RUN_MODE != "model":
        return audit["well_frames"], current_contract_table()

    frames = {well: frame.copy() for well, frame in audit["well_frames"].items()}
    catalog = audit["target_catalog"]
    contract_rows: list[dict[str, Any]] = []

    for model_target, config in TARGET_CONTRACTS.items():
        if not config["enabled"]:
            continue
        if not config["approved"]:
            raise RuntimeError(f"Target contract {model_target!r} is enabled but not approved.")
        missing_contract_fields = [
            field for field in ("reference_method", "target_definition", "source_document")
            if not clean_label(config.get(field))
        ]
        if missing_contract_fields:
            raise RuntimeError(
                f"Target contract {model_target!r} is approved but incomplete: {missing_contract_fields}. "
                "Record the method, exact definition, and source document before modeling."
            )
        if config["task"] == "classification" and not config.get("independent_from_saturation", False):
            raise RuntimeError(
                "Hydrate occurrence is enabled but independent_from_saturation is False. "
                "Occurrence labels may not be created by thresholding hydrate saturation."
            )

        selected_count = 0
        for well_alias, source_header in config["source_header_by_well"].items():
            if source_header in (None, ""):
                continue
            match = _catalog_match(catalog, well_alias, source_header)
            if match["target_family"] != config["expected_family"]:
                raise ValueError(
                    f"{well_alias}/{source_header} is cataloged as {match['target_family']}, "
                    f"not {config['expected_family']}."
                )
            frame = frames[well_alias]
            raw_key = match["raw_target_key"]
            if raw_key not in frame:
                raise KeyError(f"{raw_key} is missing from standardized frame for {well_alias}.")

            if config["task"] == "regression":
                values = numeric_series(frame[raw_key])
                finite = values.dropna()
                conversion = "identity"
                if not finite.empty and float(finite.quantile(0.99)) > 1.5 and float(finite.quantile(0.99)) <= 100.5:
                    values = values / 100.0
                    conversion = "percent_to_fraction"
                out_of_range = int(((values.dropna() < -0.001) | (values.dropna() > 1.001)).sum())
                if out_of_range:
                    raise ValueError(f"{well_alias}/{source_header} has {out_of_range} values outside [0, 1].")
                frame[model_target] = values
                class_values = ""
            else:
                labels = frame[raw_key].map(normalize_occurrence_label)
                unrecognized = sorted({value for value in labels.dropna().unique() if str(value).startswith("unrecognized::")})
                if unrecognized:
                    raise ValueError(f"Unrecognized occurrence labels for {well_alias}/{source_header}: {unrecognized[:10]}")
                allowed = {"none", "fracture", "pore"}
                observed = set(labels.dropna().unique())
                if not observed.issubset(allowed):
                    raise ValueError(f"Unsupported occurrence classes for {well_alias}: {observed}")
                frame[model_target] = labels
                conversion = "categorical_label_normalization"
                class_values = ", ".join(sorted(observed))

            selected_count += 1
            contract_rows.append({
                "model_target": model_target,
                "task": config["task"],
                "well_alias": well_alias,
                "source_header": source_header,
                "raw_target_key": raw_key,
                "target_family": match["target_family"],
                "inferred_derivation": match["inferred_derivation"],
                "required_provenance": config["required_provenance"],
                "reference_method": config.get("reference_method"),
                "target_definition": config.get("target_definition"),
                "source_document": config.get("source_document"),
                "formula_input_features": ", ".join(config.get("formula_input_features", [])),
                "labeling_evidence_features": ", ".join(config.get("labeling_evidence_features", [])),
                "conversion": conversion,
                "non_null_rows": int(frame[model_target].notna().sum()),
                "class_values": class_values,
                "approved": True,
            })

        if selected_count < 2:
            raise RuntimeError(f"Target {model_target!r} has fewer than two approved wells.")

    return frames, pd.DataFrame(contract_rows)


## 6. Paper-aligned fixed blind-well modeling

Routine development uses one split:

```text
WellA + WellB + WellC  →  WellD blind test
```

All algorithms receive the same target rows, feature panel, preprocessing, and blind well. No per-model or per-fold files are saved.

The compact benchmark includes a mean baseline, Ridge, stochastic-gradient regression, Random Forest, gradient boosting, and an ANN-style two-layer MLP. The optional `paper_12` ladder mirrors the model classes assessed by Singh et al. more closely. The MLP uses two hidden layers of 40 nodes, ReLU activation, Adam optimization, batch size 100, and up to 500 epochs; scikit-learn does not implement the source paper’s dropout layer, so the notebook labels this an ANN-style benchmark rather than an exact TensorFlow reproduction.

In [ ]:
# =============================================================================
# Model factories and shared preparation
# =============================================================================

def regression_model_factories(random_state: int) -> dict[str, Pipeline]:
    common = [("imputer", SimpleImputer(strategy="median")), ("scaler", MinMaxScaler())]
    models: dict[str, Pipeline] = {
        "mean_baseline": Pipeline(common + [("model", DummyRegressor(strategy="mean"))]),
        "ridge": Pipeline(common + [("model", Ridge(alpha=1.0))]),
        "sgd_regression": Pipeline(common + [(
            "model", SGDRegressor(
                loss="squared_error", penalty="l2", alpha=0.0001,
                max_iter=5000, tol=1e-5, random_state=random_state,
            )
        )]),
        "random_forest": Pipeline(common + [(
            "model", RandomForestRegressor(
                n_estimators=400, min_samples_leaf=3, random_state=random_state,
                n_jobs=-1,
            )
        )]),
        "gradient_boosting": Pipeline(common + [(
            "model", GradientBoostingRegressor(random_state=random_state)
        )]),
        "mlp_40_40": Pipeline(common + [(
            "model", MLPRegressor(
                hidden_layer_sizes=(40, 40), activation="relu", solver="adam",
                learning_rate_init=0.001, batch_size=100, max_iter=500,
                early_stopping=False, random_state=random_state,
            )
        )]),
    }
    if MODEL_SET == "paper_12":
        models.update({
            "kernel_ridge": Pipeline(common + [("model", KernelRidge(alpha=1.0, kernel="rbf"))]),
            "svr_rbf": Pipeline(common + [("model", SVR(kernel="rbf", C=10.0, epsilon=0.02))]),
            "decision_tree": Pipeline(common + [(
                "model", DecisionTreeRegressor(min_samples_leaf=3, random_state=random_state)
            )]),
            "adaboost": Pipeline(common + [(
                "model", AdaBoostRegressor(n_estimators=200, random_state=random_state)
            )]),
            "knn": Pipeline(common + [("model", KNeighborsRegressor(n_neighbors=15, weights="distance"))]),
            "polynomial_ridge": Pipeline(common + [
                ("poly", PolynomialFeatures(degree=2, include_bias=False)),
                ("model", Ridge(alpha=1.0)),
            ]),
            "spline_gam_like": Pipeline(common + [
                ("splines", SplineTransformer(n_knots=5, degree=3, include_bias=False)),
                ("model", Ridge(alpha=1.0)),
            ]),
        })
    return models


def classification_model_factories(random_state: int) -> dict[str, Pipeline]:
    common = [("imputer", SimpleImputer(strategy="median")), ("scaler", MinMaxScaler())]
    return {
        "majority_baseline": Pipeline(common + [("model", DummyClassifier(strategy="most_frequent"))]),
        "logistic_balanced": Pipeline(common + [(
            "model", LogisticRegression(max_iter=3000, class_weight="balanced", random_state=random_state)
        )]),
        "random_forest_balanced": Pipeline(common + [(
            "model", RandomForestClassifier(
                n_estimators=400, min_samples_leaf=3, class_weight="balanced",
                random_state=random_state, n_jobs=-1,
            )
        )]),
        "gradient_boosting": Pipeline(common + [(
            "model", GradientBoostingClassifier(random_state=random_state)
        )]),
        "svc_balanced": Pipeline(common + [(
            "model", SVC(C=10.0, kernel="rbf", class_weight="balanced", probability=True, random_state=random_state)
        )]),
        "mlp_40_40": Pipeline(common + [(
            "model", MLPClassifier(
                hidden_layer_sizes=(40, 40), activation="relu", solver="adam",
                learning_rate_init=0.001, batch_size=100, max_iter=500,
                early_stopping=False, random_state=random_state,
            )
        )]),
    }


def selected_feature_panels(target_name: str, contract: dict[str, Any]) -> dict[str, tuple[str, ...]]:
    requested = [PRIMARY_FEATURE_PANEL]
    if target_name == "hydrate_occurrence" and PRIMARY_FEATURE_PANEL not in FEATURE_PANELS:
        requested = ["occurrence_5"]
    if RUN_FEATURE_PANEL_SENSITIVITY:
        requested = list(FEATURE_PANELS)

    panels: dict[str, tuple[str, ...]] = {}
    formula_inputs = set(contract.get("formula_input_features", []))
    for name in requested:
        features = list(FEATURE_PANELS[name])
        if FEATURE_POLICY == "strict":
            features = [feature for feature in features if feature not in formula_inputs]
        if not features:
            raise RuntimeError(f"Feature panel {name!r} is empty after applying {FEATURE_POLICY!r} policy.")
        panels[name] = tuple(features)
    return panels


def _target_rows(frame: pd.DataFrame, target_name: str) -> int:
    return int(frame[target_name].notna().sum()) if target_name in frame else 0


def _balance_wells(frame: pd.DataFrame, random_state: int) -> pd.DataFrame:
    if not BALANCE_TRAINING_WELLS or frame.empty:
        return frame
    counts = frame.groupby("well_alias").size()
    if len(counts) < 2:
        return frame
    n = int(counts.min())
    parts = [
        group.sample(n=n, random_state=random_state).copy()
        for _, group in frame.groupby("well_alias", sort=False)
    ]
    return pd.concat(parts, ignore_index=True)


def _balance_classes(frame: pd.DataFrame, target_name: str, random_state: int) -> pd.DataFrame:
    if not BALANCE_OCCURRENCE_CLASSES or frame.empty:
        return frame
    counts = frame[target_name].value_counts()
    if len(counts) < 2:
        return frame
    n = int(counts.max())
    parts = [
        group.sample(n=n, replace=len(group) < n, random_state=random_state)
        for _, group in frame.groupby(target_name)
    ]
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=random_state).reset_index(drop=True)


def assemble_fixed_split(
    frames: dict[str, pd.DataFrame],
    target_name: str,
    features: tuple[str, ...],
    task: str,
    random_state: int,
) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    missing_wells = [well for well in (*TRAIN_WELLS, BLIND_WELL) if well not in frames]
    if missing_wells:
        raise KeyError(f"Configured wells are missing: {missing_wells}")

    # The feature panel is declared before seeing blind-well performance. It is
    # never changed by held-out metrics or by whichever well happens to have
    # the most complete columns. Every configured well must support it.
    coverage_failures: list[str] = []
    for feature in features:
        for well in (*TRAIN_WELLS, BLIND_WELL):
            frame = frames[well]
            coverage = float(frame[feature].notna().mean()) if feature in frame else 0.0
            if coverage < MIN_FEATURE_COVERAGE_TRAIN:
                coverage_failures.append(f"{well}:{feature}={coverage:.1%}")
    if coverage_failures:
        raise RuntimeError(
            "The predeclared feature panel is not available across the fixed split: "
            + "; ".join(coverage_failures)
        )
    usable_features = list(features)
    if len(usable_features) < 2:
        raise RuntimeError(
            f"Fewer than two predictors remain for {target_name} after leakage policy: {usable_features}"
        )

    missing_target_wells = [
        well for well in (*TRAIN_WELLS, BLIND_WELL)
        if target_name not in frames[well] or _target_rows(frames[well], target_name) < MIN_TARGET_ROWS_PER_WELL
    ]
    if missing_target_wells:
        raise RuntimeError(
            f"The fixed split requires an approved {target_name} target in every configured well. "
            f"Missing/insufficient: {missing_target_wells}"
        )

    train_parts: list[pd.DataFrame] = []
    for well in TRAIN_WELLS:
        columns = ["well_alias", "measured_depth_m", target_name, *usable_features]
        part = frames[well].reindex(columns=columns).copy()
        part = part[part[target_name].notna()]
        train_parts.append(part)
    train = pd.concat(train_parts, ignore_index=True)
    train = _balance_wells(train, random_state)
    if task == "classification":
        train = _balance_classes(train, target_name, random_state)

    blind = frames[BLIND_WELL].reindex(
        columns=["well_alias", "measured_depth_m", target_name, *usable_features]
    ).copy()
    blind = blind[blind[target_name].notna()]

    train_complete = train[usable_features].notna().mean(axis=1) >= MIN_ROW_FEATURE_FRACTION
    blind_complete = blind[usable_features].notna().mean(axis=1) >= MIN_ROW_FEATURE_FRACTION
    train = train.loc[train_complete].reset_index(drop=True)
    blind = blind.loc[blind_complete].reset_index(drop=True)
    if train.empty or blind.empty:
        raise RuntimeError(f"No scoreable rows remain for {target_name} after completeness checks.")
    return train, blind, usable_features


In [ ]:
# =============================================================================
# Evaluation runners
# =============================================================================

def regression_metrics(y_true: np.ndarray, y_pred_raw: np.ndarray) -> dict[str, float]:
    y_pred_clip = np.clip(y_pred_raw, 0.0, 1.0) if CLIP_SATURATION_TO_UNIT_INTERVAL else y_pred_raw
    return {
        "mae_raw": float(mean_absolute_error(y_true, y_pred_raw)),
        "rmse_raw": float(np.sqrt(mean_squared_error(y_true, y_pred_raw))),
        "r2_raw": float(r2_score(y_true, y_pred_raw)),
        "mae_clipped": float(mean_absolute_error(y_true, y_pred_clip)),
        "rmse_clipped": float(np.sqrt(mean_squared_error(y_true, y_pred_clip))),
        "r2_clipped": float(r2_score(y_true, y_pred_clip)),
        "bias_clipped": float(np.mean(y_pred_clip - y_true)),
        "fraction_predictions_clipped": float(np.mean((y_pred_raw < 0.0) | (y_pred_raw > 1.0))),
    }


def run_regression_target(
    frames: dict[str, pd.DataFrame],
    target_name: str,
    contract: dict[str, Any],
) -> dict[str, Any]:
    metric_rows: list[dict[str, Any]] = []
    prediction_parts: list[pd.DataFrame] = []
    fitted_models: dict[tuple[str, str, int], Pipeline] = {}

    for panel_name, requested_features in selected_feature_panels(target_name, contract).items():
        for repeat in range(N_REPEATS):
            seed = RANDOM_STATE + repeat
            train, blind, features = assemble_fixed_split(
                frames, target_name, requested_features, "regression", seed
            )
            X_train = train[features]
            y_train = pd.to_numeric(train[target_name], errors="coerce").to_numpy(dtype=float)
            X_blind = blind[features]
            y_blind = pd.to_numeric(blind[target_name], errors="coerce").to_numpy(dtype=float)

            for model_name, model in regression_model_factories(seed).items():
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    fitted = clone(model).fit(X_train, y_train)
                pred_raw = np.asarray(fitted.predict(X_blind), dtype=float)
                pred = np.clip(pred_raw, 0.0, 1.0) if CLIP_SATURATION_TO_UNIT_INTERVAL else pred_raw
                metrics = regression_metrics(y_blind, pred_raw)
                metric_rows.append({
                    "target": target_name,
                    "task": "regression",
                    "feature_panel": panel_name,
                    "model": model_name,
                    "repeat": repeat,
                    "train_wells": ",".join(TRAIN_WELLS),
                    "blind_well": BLIND_WELL,
                    "train_rows": len(train),
                    "blind_rows": len(blind),
                    "features": ",".join(features),
                    **metrics,
                })
                prediction_parts.append(pd.DataFrame({
                    "target": target_name,
                    "task": "regression",
                    "feature_panel": panel_name,
                    "model": model_name,
                    "repeat": repeat,
                    "well_alias": BLIND_WELL,
                    "depth_m": blind["measured_depth_m"].to_numpy(),
                    "reference": y_blind,
                    "prediction_raw": pred_raw,
                    "prediction": pred,
                }))
                fitted_models[(panel_name, model_name, repeat)] = fitted

    metrics_df = pd.DataFrame(metric_rows)
    ranking = (
        metrics_df.groupby(["target", "feature_panel", "model"], as_index=False)
        .agg(
            mean_rmse=("rmse_clipped", "mean"),
            std_rmse=("rmse_clipped", "std"),
            mean_mae=("mae_clipped", "mean"),
            mean_r2=("r2_clipped", "mean"),
            mean_bias=("bias_clipped", "mean"),
            repeats=("repeat", "nunique"),
        )
        .sort_values(["mean_rmse", "mean_mae"], ascending=True)
        .reset_index(drop=True)
    )
    best = ranking.iloc[0]
    best_key = (best["feature_panel"], best["model"], 0)
    selected_model = fitted_models[best_key]

    best_predictions = pd.concat(prediction_parts, ignore_index=True)
    best_predictions = best_predictions[
        (best_predictions["feature_panel"] == best["feature_panel"])
        & (best_predictions["model"] == best["model"])
        & (best_predictions["repeat"] == 0)
    ].reset_index(drop=True)

    # Recover the exact post-policy feature list used by the selected run.
    selected_row = metrics_df[
        (metrics_df["feature_panel"] == best["feature_panel"])
        & (metrics_df["model"] == best["model"])
        & (metrics_df["repeat"] == 0)
    ].iloc[0]
    requested_best_features = tuple(
        feature for feature in str(selected_row["features"]).split(",") if feature
    )

    # Permutation importance is diagnostic on the blind well and is not used to select features.
    train, blind, best_features = assemble_fixed_split(
        frames, target_name, requested_best_features, "regression", RANDOM_STATE
    )
    importance = permutation_importance(
        selected_model,
        blind[best_features],
        pd.to_numeric(blind[target_name], errors="coerce"),
        scoring="neg_root_mean_squared_error",
        n_repeats=10,
        random_state=RANDOM_STATE,
    )
    importance_df = pd.DataFrame({
        "target": target_name,
        "feature_panel": best["feature_panel"],
        "selected_model": best["model"],
        "feature": best_features,
        "importance_mean": importance.importances_mean,
        "importance_std": importance.importances_std,
    }).sort_values("importance_mean", ascending=False)

    return {
        "metrics": metrics_df,
        "ranking": ranking,
        "predictions": pd.concat(prediction_parts, ignore_index=True),
        "best_predictions": best_predictions,
        "feature_importance": importance_df,
        "selected_model": selected_model,
        "selected_model_name": best["model"],
        "selected_feature_panel": best["feature_panel"],
        "selected_features": best_features,
    }


def run_classification_target(
    frames: dict[str, pd.DataFrame],
    target_name: str,
    contract: dict[str, Any],
) -> dict[str, Any]:
    panel_name = "occurrence_5" if not RUN_FEATURE_PANEL_SENSITIVITY else PRIMARY_FEATURE_PANEL
    requested_features = FEATURE_PANELS[panel_name]
    metric_rows: list[dict[str, Any]] = []
    report_rows: list[dict[str, Any]] = []
    prediction_parts: list[pd.DataFrame] = []
    fitted_models: dict[tuple[str, int], Pipeline] = {}

    for repeat in range(N_REPEATS):
        seed = RANDOM_STATE + repeat
        train, blind, features = assemble_fixed_split(
            frames, target_name, requested_features, "classification", seed
        )
        X_train = train[features]
        y_train = train[target_name].astype(str)
        X_blind = blind[features]
        y_blind = blind[target_name].astype(str)

        for model_name, model in classification_model_factories(seed).items():
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                fitted = clone(model).fit(X_train, y_train)
            pred = fitted.predict(X_blind)
            balanced = float(balanced_accuracy_score(y_blind, pred))
            macro_f1 = float(f1_score(y_blind, pred, average="macro", zero_division=0))
            metric_rows.append({
                "target": target_name,
                "task": "classification",
                "feature_panel": panel_name,
                "model": model_name,
                "repeat": repeat,
                "train_wells": ",".join(TRAIN_WELLS),
                "blind_well": BLIND_WELL,
                "train_rows": len(train),
                "blind_rows": len(blind),
                "features": ",".join(features),
                "balanced_accuracy": balanced,
                "macro_f1": macro_f1,
            })
            report = classification_report(y_blind, pred, output_dict=True, zero_division=0)
            for label in ("none", "fracture", "pore"):
                values = report.get(label, {})
                report_rows.append({
                    "target": target_name,
                    "model": model_name,
                    "repeat": repeat,
                    "class": label,
                    "precision": values.get("precision", np.nan),
                    "recall": values.get("recall", np.nan),
                    "f1": values.get("f1-score", np.nan),
                    "support": values.get("support", 0),
                })
            prediction_parts.append(pd.DataFrame({
                "target": target_name,
                "task": "classification",
                "feature_panel": panel_name,
                "model": model_name,
                "repeat": repeat,
                "well_alias": BLIND_WELL,
                "depth_m": blind["measured_depth_m"].to_numpy(),
                "reference": y_blind.to_numpy(),
                "prediction": pred,
            }))
            fitted_models[(model_name, repeat)] = fitted

    metrics_df = pd.DataFrame(metric_rows)
    ranking = (
        metrics_df.groupby(["target", "feature_panel", "model"], as_index=False)
        .agg(
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            std_balanced_accuracy=("balanced_accuracy", "std"),
            mean_macro_f1=("macro_f1", "mean"),
            repeats=("repeat", "nunique"),
        )
        .sort_values(["mean_balanced_accuracy", "mean_macro_f1"], ascending=False)
        .reset_index(drop=True)
    )
    best = ranking.iloc[0]
    selected_model = fitted_models[(best["model"], 0)]
    all_predictions = pd.concat(prediction_parts, ignore_index=True)
    best_predictions = all_predictions[
        (all_predictions["model"] == best["model"])
        & (all_predictions["repeat"] == 0)
    ].reset_index(drop=True)

    selected_row = metrics_df[
        (metrics_df["feature_panel"] == best["feature_panel"])
        & (metrics_df["model"] == best["model"])
        & (metrics_df["repeat"] == 0)
    ].iloc[0]
    selected_features = [
        feature for feature in str(selected_row["features"]).split(",") if feature
    ]

    return {
        "metrics": metrics_df,
        "ranking": ranking,
        "class_report": pd.DataFrame(report_rows),
        "predictions": all_predictions,
        "best_predictions": best_predictions,
        "selected_model": selected_model,
        "selected_model_name": best["model"],
        "selected_feature_panel": best["feature_panel"],
        "selected_features": selected_features,
    }


## 7. Consolidated results and one selected deployment model

Model selection uses the fixed blind well. After selection, the chosen algorithm may be refit on all approved labeled wells for deployment. That refit is not a new validation result; it is the model intended for future unlabeled wells.

The full well rotation remains optional and is not persisted as separate models. When enabled for a final audit, only an aggregated metrics sheet is retained.

In [ ]:
# =============================================================================
# Final refit, figures, and consolidated export
# =============================================================================

def refit_selected_model(
    frames: dict[str, pd.DataFrame],
    target_name: str,
    task: str,
    selected_model: Pipeline,
    selected_features: list[str],
) -> Pipeline:
    parts: list[pd.DataFrame] = []
    for well_alias, frame in frames.items():
        if target_name not in frame or frame[target_name].notna().sum() < MIN_TARGET_ROWS_PER_WELL:
            continue
        part = frame.reindex(columns=["well_alias", target_name, *selected_features]).copy()
        part = part[part[target_name].notna()]
        part = part[part[selected_features].notna().mean(axis=1) >= MIN_ROW_FEATURE_FRACTION]
        parts.append(part)
    all_data = pd.concat(parts, ignore_index=True)
    all_data = _balance_wells(all_data, RANDOM_STATE)
    if task == "classification":
        all_data = _balance_classes(all_data, target_name, RANDOM_STATE)
    final = clone(selected_model)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        final.fit(all_data[selected_features], all_data[target_name])
    return final


def create_figures(results_by_target: dict[str, dict[str, Any]]) -> None:
    with PdfPages(FIGURES_PDF) as pdf:
        for target_name, result in results_by_target.items():
            predictions = result["best_predictions"].copy()
            task = predictions["task"].iloc[0]
            if task == "regression":
                y_true = pd.to_numeric(predictions["reference"], errors="coerce")
                y_pred = pd.to_numeric(predictions["prediction"], errors="coerce")

                fig, ax = plt.subplots(figsize=(6.5, 6.0))
                ax.scatter(y_true, y_pred, s=12, alpha=0.55)
                lo = float(np.nanmin([y_true.min(), y_pred.min(), 0.0]))
                hi = float(np.nanmax([y_true.max(), y_pred.max(), 1.0]))
                ax.plot([lo, hi], [lo, hi], linestyle="--", linewidth=1)
                ax.set_xlabel("Reference hydrate saturation")
                ax.set_ylabel("Predicted hydrate saturation")
                ax.set_title(f"Blind well {BLIND_WELL}: {result['selected_model_name']}")
                ax.text(
                    0.03, 0.97,
                    f"RMSE={np.sqrt(mean_squared_error(y_true, y_pred)):.3f}\nR²={r2_score(y_true, y_pred):.3f}\nn={len(y_true)}",
                    transform=ax.transAxes, va="top",
                )
                fig.tight_layout()
                pdf.savefig(fig)
                plt.close(fig)

                fig, ax = plt.subplots(figsize=(6.5, 8.0))
                order = np.argsort(predictions["depth_m"].to_numpy())
                depth = predictions["depth_m"].to_numpy()[order]
                ax.plot(y_true.to_numpy()[order], depth, label="Reference")
                ax.plot(y_pred.to_numpy()[order], depth, label="Prediction")
                ax.invert_yaxis()
                ax.set_xlabel("Hydrate saturation")
                ax.set_ylabel("Measured depth (m)")
                ax.set_title(f"Blind-well depth profile: {BLIND_WELL}")
                ax.legend()
                fig.tight_layout()
                pdf.savefig(fig)
                plt.close(fig)

                ranking = result["ranking"].sort_values("mean_rmse")
                fig, ax = plt.subplots(figsize=(8.0, 5.0))
                ax.barh(ranking["model"], ranking["mean_rmse"])
                ax.invert_yaxis()
                ax.set_xlabel("Blind-well RMSE")
                ax.set_title(f"Model comparison — {target_name}")
                fig.tight_layout()
                pdf.savefig(fig)
                plt.close(fig)
            else:
                labels = ["none", "fracture", "pore"]
                cm = confusion_matrix(predictions["reference"], predictions["prediction"], labels=labels)
                fig, ax = plt.subplots(figsize=(6.5, 5.5))
                image = ax.imshow(cm)
                ax.set_xticks(range(len(labels)), labels=labels, rotation=30, ha="right")
                ax.set_yticks(range(len(labels)), labels=labels)
                ax.set_xlabel("Predicted")
                ax.set_ylabel("Reference")
                ax.set_title(f"Occurrence confusion matrix — {BLIND_WELL}")
                for i in range(cm.shape[0]):
                    for j in range(cm.shape[1]):
                        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
                fig.colorbar(image, ax=ax)
                fig.tight_layout()
                pdf.savefig(fig)
                plt.close(fig)


def build_run_summary(
    contract_audit: pd.DataFrame,
    results_by_target: dict[str, dict[str, Any]],
) -> pd.DataFrame:
    rows = [
        {"item": "created_utc", "value": datetime.now(timezone.utc).isoformat()},
        {"item": "run_mode", "value": RUN_MODE},
        {"item": "data_directory", "value": str(DATA_DIR)},
        {"item": "train_wells", "value": ", ".join(TRAIN_WELLS)},
        {"item": "blind_well", "value": BLIND_WELL},
        {"item": "model_set", "value": MODEL_SET},
        {"item": "n_repeats", "value": N_REPEATS},
        {"item": "feature_policy", "value": FEATURE_POLICY},
        {"item": "primary_feature_panel", "value": PRIMARY_FEATURE_PANEL},
        {"item": "validation_statement", "value": "Metrics are from one fixed blind well; final refits are deployment models, not validation."},
    ]
    for target_name, result in results_by_target.items():
        rows.extend([
            {"item": f"{target_name}__selected_model", "value": result["selected_model_name"]},
            {"item": f"{target_name}__selected_feature_panel", "value": result["selected_feature_panel"]},
            {"item": f"{target_name}__selected_features", "value": ", ".join(result["selected_features"])},
        ])
    for name, reference in PAPER_REFERENCES.items():
        rows.append({"item": f"reference__{name}", "value": reference})
    return pd.DataFrame(rows)


def export_model_results(
    audit: dict[str, Any],
    contract_audit: pd.DataFrame,
    results_by_target: dict[str, dict[str, Any]],
    deployment_bundle: dict[str, Any],
) -> None:
    prepare_known_output_paths()
    summary = build_run_summary(contract_audit, results_by_target)

    metrics = pd.concat([result["metrics"] for result in results_by_target.values()], ignore_index=True, sort=False)
    rankings = pd.concat([result["ranking"] for result in results_by_target.values()], ignore_index=True, sort=False)
    predictions = pd.concat([result["predictions"] for result in results_by_target.values()], ignore_index=True, sort=False)
    best_predictions = pd.concat([result["best_predictions"] for result in results_by_target.values()], ignore_index=True, sort=False)
    importance_parts = [
        result.get("feature_importance", pd.DataFrame())
        for result in results_by_target.values()
        if not result.get("feature_importance", pd.DataFrame()).empty
    ]
    class_report_parts = [
        result.get("class_report", pd.DataFrame())
        for result in results_by_target.values()
        if not result.get("class_report", pd.DataFrame()).empty
    ]
    feature_importance = (
        pd.concat(importance_parts, ignore_index=True, sort=False)
        if importance_parts else pd.DataFrame()
    )
    class_reports = (
        pd.concat(class_report_parts, ignore_index=True, sort=False)
        if class_report_parts else pd.DataFrame()
    )

    with pd.ExcelWriter(RESULTS_XLSX, engine="openpyxl") as writer:
        summary.to_excel(writer, sheet_name="run_summary", index=False)
        contract_audit.to_excel(writer, sheet_name="target_contract", index=False)
        audit["target_catalog"].to_excel(writer, sheet_name="target_catalog", index=False)
        audit["target_pair_checks"].to_excel(writer, sheet_name="target_pair_checks", index=False)
        audit["paper_target_methods"].to_excel(writer, sheet_name="paper_target_methods", index=False)
        audit["source_layout"].to_excel(writer, sheet_name="source_layout", index=False)
        audit["feature_mapping"].to_excel(writer, sheet_name="feature_mapping", index=False)
        audit["unit_audit"].to_excel(writer, sheet_name="unit_audit", index=False)
        audit["well_readiness"].to_excel(writer, sheet_name="well_readiness", index=False)
        metrics.to_excel(writer, sheet_name="model_metrics", index=False)
        rankings.to_excel(writer, sheet_name="model_ranking", index=False)
        feature_importance.to_excel(writer, sheet_name="feature_importance", index=False)
        class_reports.to_excel(writer, sheet_name="class_metrics", index=False)
        best_predictions.head(10000).to_excel(writer, sheet_name="prediction_sample", index=False)

    predictions.to_parquet(PREDICTIONS_PARQUET, index=False)
    create_figures(results_by_target)
    joblib.dump(deployment_bundle, SELECTED_MODELS)


def run_modeling(audit: dict[str, Any]) -> dict[str, Any]:
    if RUN_MODE != "model":
        print("RUN_MODE is 'audit'; no models were trained.")
        return {"audit": audit}

    frames, contract_audit = resolve_target_contracts(audit)
    results_by_target: dict[str, dict[str, Any]] = {}
    deployment_models: dict[str, Any] = {}

    for target_name, contract in TARGET_CONTRACTS.items():
        if not contract["enabled"]:
            continue
        if contract["task"] == "regression":
            result = run_regression_target(frames, target_name, contract)
        else:
            result = run_classification_target(frames, target_name, contract)
        results_by_target[target_name] = result

        deployment_model = result["selected_model"]
        if REFIT_SELECTED_MODEL_ON_ALL_LABELED_WELLS:
            deployment_model = refit_selected_model(
                frames,
                target_name,
                contract["task"],
                result["selected_model"],
                list(result["selected_features"]),
            )
        deployment_models[target_name] = {
            "model": deployment_model,
            "task": contract["task"],
            "selected_algorithm": result["selected_model_name"],
            "feature_panel": result["selected_feature_panel"],
            "features": list(result["selected_features"]),
            "target_contract": contract,
            "refit_on_all_labeled_wells": REFIT_SELECTED_MODEL_ON_ALL_LABELED_WELLS,
        }

    deployment_bundle = {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "models": deployment_models,
        "train_wells": TRAIN_WELLS,
        "blind_well_used_for_selection": BLIND_WELL,
        "paper_references": PAPER_REFERENCES,
    }
    export_model_results(audit, contract_audit, results_by_target, deployment_bundle)

    print("\nMODEL RANKINGS")
    for target_name, result in results_by_target.items():
        print("\n", target_name)
        display(result["ranking"])
    print("\nConsolidated results:", RESULTS_XLSX)
    print("Predictions:", PREDICTIONS_PARQUET)
    print("Figures:", FIGURES_PDF)
    print("Selected deployment models:", SELECTED_MODELS)
    return {
        "audit": audit,
        "contract_audit": contract_audit,
        "results": results_by_target,
        "deployment_bundle": deployment_bundle,
    }


## 8. Run status

With the default configuration, this final cell confirms that the target audit completed and that modeling remains blocked. After the target contracts are completed and approved, rerunning the notebook will train the fixed blind-well benchmark and write the four consolidated files.

In [ ]:
results = run_modeling(audit)

if RUN_MODE == "audit":
    print("\nNEXT ACTION")
    print("1. Open model_results.xlsx and review target_catalog and target_pair_checks.")
    print("2. Confirm each target's equation/provenance in the original workbook or source documentation.")
    print("3. Fill TARGET_CONTRACTS with exact source headers and direct formula inputs.")
    print("4. Set approved=True and RUN_MODE='model'.")


## Interpretation rules

- A strong blind-well score does not prove that similarly named target columns are equivalent.
- A model trained on a target derived from density porosity must flag density-derived predictors as circularity-sensitive.
- Occurrence and saturation are separate outputs. Occurrence labels cannot be created by applying a cutoff to saturation.
- The one fixed blind-well split is the routine development result. Enable repeated runs or full well rotation only after the scientific design is frozen.
- The final refit on all labeled wells is a deployment model and must not be reported as an independent validation score.
- Water/residual saturation columns remain visible in the audit for provenance review but are excluded from model targets.

### Target meaning guardrail

- **NMR-density saturation** and **Archie-derived saturation** are different reference products even when both are written as `Sh`.
- `S_wr`, `Swr`, and similar water/residual-water quantities remain audit-only and are not project ML responses.
- Occurrence must come from an independent interpretation record. A threshold applied to saturation is not an occurrence target.
- The default strict feature policy removes any predictor explicitly listed as a direct input to the approved saturation formula; paper-replication sensitivity may be run separately and must be labeled as such.
